# Technical Indicators — Feature Engineering Pipeline

Loads `data/legacy_candles.jsonl`, computes all indicators per snapshot
(rolling, simulating real-time inference), and writes `data/legacy_features.jsonl`.

Each output row = snapshot fields + 33 indicators + bet outcome.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from technicals import compute_all
from tqdm.notebook import tqdm

## 1. Load and sort data

In [ ]:
DATA_PATH = Path("../data/legacy_candles.jsonl")
OUT_PATH = Path("../data/legacy_features.jsonl")

candles = []
with open(DATA_PATH) as f:
    for line in f:
        candles.append(json.loads(line))

candles.sort(key=lambda c: c["start_time"])
print(f"Loaded {len(candles)} candles, {sum(len(c['snapshots']) for c in candles):,} snapshots")

## 2. Compute indicators per snapshot

In [ ]:
rows = []
prior_candles = []

for candle in tqdm(candles, desc="Candles"):
    snapshots = candle["snapshots"]
    outcome = candle["outcome"]
    candle_open = candle["open"]

    for si in range(len(snapshots)):
        snapshots_so_far = snapshots[: si + 1]
        indicators = compute_all(prior_candles, candle_open, snapshots_so_far)

        # Snapshot fields (drop nested lists to keep rows flat)
        snap = snapshots[si]
        row = {
            "candle_id": candle["candle_id"],
            "timestamp": snap["timestamp"],
            "elapsed_pct": snap["elapsed_pct"],
            "btc_price": snap["btc_price"],
            "up_best_bid": snap["up_bids"][0][0] if snap["up_bids"] else None,
            "up_best_ask": snap["up_asks"][0][0] if snap["up_asks"] else None,
            "up_bid_depth": snap["up_bids"][0][1] if snap["up_bids"] else None,
            "up_ask_depth": snap["up_asks"][0][1] if snap["up_asks"] else None,
            "down_best_bid": snap["down_bids"][0][0] if snap["down_bids"] else None,
            "down_best_ask": snap["down_asks"][0][0] if snap["down_asks"] else None,
            "down_bid_depth": snap["down_bids"][0][1] if snap["down_bids"] else None,
            "down_ask_depth": snap["down_asks"][0][1] if snap["down_asks"] else None,
            "market_volume": snap["market_volume"],
            **indicators,
            "outcome": outcome,
        }
        rows.append(row)

    # This candle is now "prior" for next ones
    prior_candles.append(candle)

print(f"\nDone: {len(rows):,} rows from {len(candles)} candles")

## 3. Write output JSONL

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, "w") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

print(f"Wrote {len(rows):,} rows to {OUT_PATH}")
print(f"File size: {OUT_PATH.stat().st_size / 1024 / 1024:.1f} MB")

## 4. Coverage & quality check

In [ ]:
# Check how many rows have non-null values per indicator
indicator_keys = [
    k
    for k in rows[0]
    if k
    not in (
        "candle_id",
        "timestamp",
        "elapsed_pct",
        "btc_price",
        "up_best_bid",
        "up_best_ask",
        "up_bid_depth",
        "up_ask_depth",
        "down_best_bid",
        "down_best_ask",
        "down_bid_depth",
        "down_ask_depth",
        "market_volume",
        "outcome",
    )
]

print(f"{'Indicator':<30} {'Non-null':>8} {'Coverage':>8}")
print("-" * 50)
for key in indicator_keys:
    non_null = sum(1 for r in rows if r[key] is not None)
    pct = non_null / len(rows) * 100
    print(f"{key:<30} {non_null:>8,} {pct:>7.1f}%")

In [ ]:
# Outcome distribution in output
up = sum(1 for r in rows if r["outcome"] == "UP")
dn = len(rows) - up
print(f"UP snapshots: {up:,} ({up / len(rows) * 100:.1f}%)")
print(f"DOWN snapshots: {dn:,} ({dn / len(rows) * 100:.1f}%)")

In [ ]:
# Sample output row
rows[5000]

## 5. Indicator distributions

In [ ]:
def plot_indicator(name, rows):
    up_vals = [r[name] for r in rows if r[name] is not None and r["outcome"] == "UP"]
    dn_vals = [r[name] for r in rows if r[name] is not None and r["outcome"] == "DOWN"]
    if not up_vals or not dn_vals:
        print(f"Skipping {name} — no data")
        return

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.hist(up_vals, bins=60, alpha=0.6, color="#2ecc71", label="UP", density=True)
    ax.hist(dn_vals, bins=60, alpha=0.6, color="#e74c3c", label="DOWN", density=True)
    ax.set_title(name)
    ax.legend()
    plt.tight_layout()
    plt.show()


for key in indicator_keys:
    plot_indicator(key, rows)

## 6. Effect size ranking

In [ ]:
ranked = []
for key in indicator_keys:
    up_v = [r[key] for r in rows if r[key] is not None and r["outcome"] == "UP"]
    dn_v = [r[key] for r in rows if r[key] is not None and r["outcome"] == "DOWN"]
    if not up_v or not dn_v:
        continue
    all_v = up_v + dn_v
    diff = np.mean(up_v) - np.mean(dn_v)
    std = np.std(all_v)
    effect = abs(diff) / std if std > 0 else 0.0
    ranked.append((key, len(all_v), np.mean(up_v), np.mean(dn_v), diff, effect))

ranked.sort(key=lambda x: -x[5])

print(f"{'Indicator':<30} {'N':>8} {'UP mean':>10} {'DN mean':>10} {'Effect':>8}")
print("-" * 70)
for name, n, up_m, dn_m, _diff, effect in ranked:
    print(f"{name:<30} {n:>8,} {up_m:>10.4f} {dn_m:>10.4f} {effect:>8.4f}")

In [ ]:
names = [r[0] for r in ranked]
effects = [r[5] for r in ranked]

fig, ax = plt.subplots(figsize=(10, 10))
colors = ["#2ecc71" if e > 0.1 else "#f39c12" if e > 0.05 else "#95a5a6" for e in effects]
ax.barh(range(len(names)), effects, color=colors, edgecolor="white")
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("Effect Size (|mean diff| / std)")
ax.set_title("Indicator Predictive Power")
ax.invert_yaxis()
ax.axvline(0.1, color="green", linestyle="--", alpha=0.5, label="0.1")
ax.legend()
plt.tight_layout()
plt.show()